# 多层感知机训练示例

本笔记本演示如何根据 PPT 中给出的结构训练一个多层感知机（MLP），用于根据当前油面高度和未匹配负载阻抗预测三支节目标油面高度。模型的结构如下：

- 输入层：5 个特征，分别为 `ZR`, `ZI`, `L10`, `L20`, `L30`
- 隐藏层：第一层 128 个神经元，第二层 64 个神经元，激活函数均为 `tanh`
- 输出层：3 个神经元，通过 `7 * sigmoid` 变换将输出限制在 0-7 之间

> **数据格式要求**
>
> - 训练集 CSV 必须包含下列列：`ZR`, `ZI`, `L10`, `L20`, `L30`, `L1`, `L2`, `L3`
>   - 前五列作为网络输入。
>   - `L1`, `L2`, `L3` 为目标油面高度标签，取值范围 [0, 7]。
> - 测试集 CSV 至少需要包含列：`ZR`, `ZI`, `L10`, `L20`, `L30`, `L1`, `L2`, `L3`。
>   - 如果测试集中不包含标签列 `L1`, `L2`, `L3`，可以将下面评估部分中使用真实标签的代码注释掉。
>
> 文件示例（CSV 带表头）：
>
> ```csv
> ZR,ZI,L10,L20,L30,L1,L2,L3
> 15.2,-3.8,2.1,4.3,1.6,2.0,4.0,2.0
> ...
> ```

在运行前，请将训练集与测试集 CSV 文件的路径填写在对应的变量中。

In [ ]:
# 导入所需库
import os
import importlib.util
from typing import Tuple

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

MATPLOTLIB_SPEC = importlib.util.find_spec("matplotlib")
HAS_MATPLOTLIB = MATPLOTLIB_SPEC is not None
if HAS_MATPLOTLIB:
    import matplotlib
    matplotlib.use("Agg", force=True)
    import matplotlib.pyplot as plt
else:
    plt = None

print(f"PyTorch version: {torch.__version__}")


In [ ]:
# TODO: 将以下路径替换为您本地的训练/测试 CSV 文件路径
TRAIN_CSV_PATH = "./data/train.csv"  # 训练集 CSV
TEST_CSV_PATH = "./data/test.csv"    # 测试集 CSV

if not os.path.exists(TRAIN_CSV_PATH):
    print(f"警告：训练集 {TRAIN_CSV_PATH} 当前不存在，请先准备数据文件。")
if not os.path.exists(TEST_CSV_PATH):
    print(f"警告：测试集 {TEST_CSV_PATH} 当前不存在，请先准备数据文件。")

In [ ]:
class OilLevelDataset(Dataset):
    """用于加载油面调节数据集的 Dataset。"""

    def __init__(self, csv_path: str):
        df = pd.read_csv(csv_path)
        required_columns = ["ZR", "ZI", "L10", "L20", "L30", "L1", "L2", "L3"]
        missing = [col for col in required_columns if col not in df.columns]
        if missing:
            raise ValueError(f"CSV 缺少必要列: {missing}")

        self.inputs = df[["ZR", "ZI", "L10", "L20", "L30"]].astype(np.float32).values
        self.targets = df[["L1", "L2", "L3"]].astype(np.float32).values

    def __len__(self) -> int:
        return len(self.inputs)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        x = torch.from_numpy(self.inputs[idx])
        y = torch.from_numpy(self.targets[idx])
        return x, y

def load_datasets(train_csv: str, test_csv: str):
    train_dataset = OilLevelDataset(train_csv)
    test_dataset = OilLevelDataset(test_csv)
    return train_dataset, test_dataset

train_dataset, test_dataset = load_datasets(TRAIN_CSV_PATH, TEST_CSV_PATH)
print(f"训练样本数: {len(train_dataset)}")
print(f"测试样本数: {len(test_dataset)}")

In [ ]:
class OilLevelMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(5, 128),
            nn.Tanh(),
            nn.Linear(128, 64),
            nn.Tanh(),
            nn.Linear(64, 3),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 输出范围 [0, 1]，再乘以 7 将范围缩放到 [0, 7]
        return 7 * self.net(x)

model = OilLevelMLP()
print(model)

In [ ]:
class MinimalAdjustmentLoss(nn.Module):
    """自定义损失函数 = MSE + alpha * 平均动作幅度"""

    def __init__(self, alpha: float = 0.1):
        super().__init__()
        self.alpha = alpha
        self.mse = nn.MSELoss()

    def forward(self, predictions: torch.Tensor, targets: torch.Tensor, current_levels: torch.Tensor) -> torch.Tensor:
        mse_loss = self.mse(predictions, targets)
        action_penalty = torch.mean(torch.abs(predictions - current_levels))
        return mse_loss + self.alpha * action_penalty

loss_fn = MinimalAdjustmentLoss(alpha=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

BATCH_SIZE = 32
EPOCHS = 200

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
def train_epoch(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer, loss_fn: MinimalAdjustmentLoss) -> float:
    model.train()
    total_loss = 0.0
    for inputs, targets in loader:
        optimizer.zero_grad()
        preds = model(inputs)
        current_levels = inputs[:, 2:5]
        loss = loss_fn(preds, targets, current_levels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(inputs)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, loss_fn: MinimalAdjustmentLoss) -> Tuple[float, float]:
    model.eval()
    total_loss = 0.0
    total_mae = 0.0
    for inputs, targets in loader:
        preds = model(inputs)
        current_levels = inputs[:, 2:5]
        loss = loss_fn(preds, targets, current_levels)
        total_loss += loss.item() * len(inputs)
        total_mae += torch.mean(torch.abs(preds - targets)).item() * len(inputs)
    avg_loss = total_loss / len(loader.dataset)
    avg_mae = total_mae / len(loader.dataset)
    return avg_loss, avg_mae

train_history = []
test_history = []

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, optimizer, loss_fn)
    test_loss, test_mae = evaluate(model, test_loader, loss_fn)
    train_history.append(train_loss)
    test_history.append((test_loss, test_mae))
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f} | Test MAE: {test_mae:.4f}")

In [ ]:
from pathlib import Path

PLOT_DIR = Path('./plots')
PLOT_DIR.mkdir(parents=True, exist_ok=True)

train_losses = [loss for loss in train_history]
test_losses = [loss for loss, _ in test_history]
test_maes = [mae for _, mae in test_history]
epoch_indices = list(range(1, EPOCHS + 1))

import pandas as pd
metrics_df = pd.DataFrame({
    'epoch': epoch_indices,
    'train_loss': train_losses,
    'test_loss': test_losses,
    'test_mae': test_maes,
})
metrics_path = PLOT_DIR / 'training_metrics.csv'
metrics_df.to_csv(metrics_path, index=False)
print(f'训练与验证指标已保存至 {metrics_path.resolve()}')

if HAS_MATPLOTLIB and plt is not None:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(epoch_indices, train_losses, label='Train Loss')
    ax.plot(epoch_indices, test_losses, label='Test Loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Training and Validation Loss')
    ax.legend()
    fig.tight_layout()
    fig.savefig(PLOT_DIR / 'loss_curves.png', dpi=150)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(epoch_indices, test_maes, label='Test MAE')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Mean Absolute Error')
    ax.set_title('Test Mean Absolute Error')
    ax.legend()
    fig.tight_layout()
    fig.savefig(PLOT_DIR / 'test_mae.png', dpi=150)
    plt.close(fig)
    print(f'学习曲线图像已保存至 {PLOT_DIR.resolve()}')
else:
    print('检测到未安装 matplotlib 或当前环境禁用图形渲染，仅保存了数值指标。')


In [ ]:
@torch.no_grad()
def predict(model: nn.Module, csv_path: str) -> pd.DataFrame:
    dataset = OilLevelDataset(csv_path)
    loader = DataLoader(dataset, batch_size=64, shuffle=False)
    preds = []
    for inputs, _ in loader:
        outputs = model(inputs)
        preds.append(outputs.detach().cpu().numpy())
    preds = np.vstack(preds)
    df = pd.read_csv(csv_path).copy()
    df[["Pred_L1", "Pred_L2", "Pred_L3"]] = preds
    return df

# torch.save(model.state_dict(), "oil_level_mlp.pth")
# predictions = predict(model, TEST_CSV_PATH)
# predictions.to_csv("predictions.csv", index=False)
# predictions.head()


In [ ]:
RESULTS_DIR = Path('./results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

test_predictions = predict(model, TEST_CSV_PATH)
comparison_columns = ["L1", "L2", "L3", "Pred_L1", "Pred_L2", "Pred_L3"]
print('测试集实际值与预测值预览：')
print(test_predictions[comparison_columns].head())

comparison_path = RESULTS_DIR / 'test_predictions_vs_actual.csv'
test_predictions.to_csv(comparison_path, index=False)
print(f'测试集完整对比结果已保存至 {comparison_path.resolve()}')

targets = test_predictions[["L1", "L2", "L3"]].to_numpy(dtype=np.float32)
preds = test_predictions[["Pred_L1", "Pred_L2", "Pred_L3"]].to_numpy(dtype=np.float32)
errors = preds - targets

mse = float(np.mean(errors ** 2))
rmse = float(np.sqrt(mse))
mae = float(np.mean(np.abs(errors)))

ss_res = np.sum(errors ** 2, axis=0)
ss_tot = np.sum((targets - targets.mean(axis=0)) ** 2, axis=0)
with np.errstate(divide='ignore', invalid='ignore'):
    r2_per_target = 1 - np.divide(ss_res, ss_tot, out=np.zeros_like(ss_res), where=ss_tot != 0)
r2 = float(np.mean(r2_per_target))

print(f'均方误差 (MSE): {mse:.4f}')
print(f'均方根误差 (RMSE): {rmse:.4f}')
print(f'平均绝对误差 (MAE): {mae:.4f}')
print(f'决定系数 (R^2): {r2:.4f}')
